# Conversation Agent Demo

This notebook shows a minimal conversation agent that uses only working memory.

It uses:
- `WorkingMemory`
- `MemoryRetrieveNode`
- one custom `ConversationNode(PlannerNode)`
- `ResponseNode`
- `MemoryNode`
- a local Qwen model for the general conversation path

There are no tools, no ReAct loop, and no long-term memory layers in this example.


In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current notebook path.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

required_packages = ["langgraph", "transformers", "accelerate", "sentencepiece"]
for package in required_packages:
    try:
        __import__(package)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print(f"Using repository root: {repo_root}")
print(f"Python executable: {sys.executable}")


Using repository root: /Users/saketm10/Projects/openclaw_agents
Python executable: /Users/saketm10/Projects/openclaw_agents/.venv/bin/python3.11


In [2]:
import re
from types import SimpleNamespace
from langgraph.graph import END, START, StateGraph

from src.agents.nodes import MemoryNode, MemoryRetrieveNode, PlannerNode, ResponseNode
from src.agents.nodes.types import AgentState
from src.llm.qwen import Qwen3_1_7BLLM
from src.memory import WorkingMemory
from src.tools.registry import ToolRegistry


class ConversationNode(PlannerNode):
    def plan(
        self,
        *,
        user_input: str,
        memory=None,
        observation=None,
        memory_context=None,
        system_prompt=None,
        user_prompt=None,
        available_tools=None,
    ):
        del observation, available_tools
        assert memory is not None
        working_memory = (memory_context or {}).get("working", memory)

        match = re.search(r"my name is\s+([A-Za-z][A-Za-z\s'-]*)", user_input, re.IGNORECASE)
        if match:
            name = match.group(1).strip()
            return SimpleNamespace(
                thought="Stored the user name.",
                tool_call=None,
                memory_updates=[{"target": "working", "operation": "set_state", "values": {"user_name": name}}],
                respond_directly=True,
                response_text="I'll remember that.",
                done=True,
            )

        if "what is my name" in user_input.lower() or "what's my name" in user_input.lower():
            name = working_memory.state.get("user_name")
            if name:
                return SimpleNamespace(
                    thought="Answered from working memory.",
                    tool_call=None,
                    respond_directly=True,
                    response_text=f"Your name is {name}.",
                    done=True,
                )
            return SimpleNamespace(
                thought="No name stored in working memory.",
                tool_call=None,
                respond_directly=True,
                response_text="You have not told me your name yet.",
                done=True,
            )

        return super().plan(
            user_input=user_input,
            memory=memory,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
        )


qwen_llm = Qwen3_1_7BLLM(model_name="Qwen/Qwen3-1.7B", max_new_tokens=128, enable_thinking=False)
memory = WorkingMemory(session_id="demo-session")
memory_retrieve_node = MemoryRetrieveNode(
    tool_registry=ToolRegistry(),
    memories=[WorkingMemory],
)
conversation_node = ConversationNode(
    llm=qwen_llm,
    system_prompt="You are a concise conversation assistant.",
    user_prompt="User: {user_input}",
)
response_node = ResponseNode(default_response="I do not know how to respond.")
memory_node = MemoryNode(memories=[memory])

graph = StateGraph(AgentState)
graph.add_node("retrieve_memory", memory_retrieve_node.execute)
graph.add_node("plan", conversation_node.execute)
graph.add_node("respond", response_node.execute)
graph.add_node("memory", memory_node.execute)
graph.add_edge(START, "retrieve_memory")
graph.add_edge("retrieve_memory", "plan")
graph.add_conditional_edges(
    "plan",
    conversation_node.route,
    {"respond": "respond", "end": END, "act": "respond"},
)
graph.add_edge("respond", "memory")
graph.add_edge("memory", END)
compiled_graph = graph.compile()


In [3]:
state = compiled_graph.invoke(
    {
        "user_input": "My name is Saket",
        "memory": memory,
        "steps": 0,
    }
)
state["response"]


"I'll remember that."

In [4]:
state = compiled_graph.invoke(
    {
        "user_input": "What is my name?",
        "memory": memory,
        "steps": 0,
    }
)
state["response"]


'Your name is Saket.'

In [5]:
memory.state


{'node_llms': {'planner': 'Qwen/Qwen3-1.7B'},
 'current_goal': 'What is my name?',
 'user_name': 'Saket'}

In [6]:
[(item["role"], item["content"]) for item in memory.recent_items]


[('user', 'My name is Saket'),
 ('agent', "I'll remember that."),
 ('user', 'What is my name?'),
 ('agent', 'Your name is Saket.'),
 ('user', 'What is my name?'),
 ('agent', 'Your name is Saket.')]